# FDA Food Recall — Exploratory Data Analysis & Cleaning

**Purpose:** Explore and clean raw FDA food recall data pulled from the openFDA API, prior
to building a Power BI dashboard. This notebook documents every cleaning decision so the
dashboard's numbers are fully traceable back to raw source data.

**Data source:** openFDA Food Enforcement API — https://api.fda.gov/food/enforcement.json
**Records pulled:** 1,000 (API max per call; 29,171 total available historically)


In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)

df = pd.read_csv("data/fda_recalls_raw.csv")
df.columns = [c.strip() for c in df.columns]
print(f"Shape: {df.shape}")
df.head(3)

Shape: (1000, 24)


,results.address_1,results.address_2,Sum of results.center_classification_date,results.city,results.classification,results.code_info,results.country,results.distribution_pattern,Sum of results.event_id,results.initial_firm_notification,results.more_code_info,results.product_description,results.postal_code,results.product_quantity,results.product_type,results.reason_for_recall,Sum of results.recall_initiation_date,results.recall_number,results.recalling_firm,Sum of results.report_date,results.state,results.status,Sum of results.termination_date,results.voluntary_mandated
0,#10 Pictsweet Drive,NaN,20160707,Bells,Class I,969 cases of 8 units: Best if Used by 3/28/2...,United States,nationwide,73853,E-Mail,NaN,"PICTSWEET(R) Steamables, Seasoned Spring Veget...",38006,79932 units,Food,Products contain onions which were recalled fo...,20160409,F-1659-2016,The Pictsweet Company,20160713,TN,Terminated,20170803.0,Voluntary: Firm initiated
1,#10 Pictsweet Drive,NaN,20160707,Bells,Class I,"a) UPC 11110 89791: Lot Codes: 0896BG, 0896BH...",United States,nationwide,73853,E-Mail,NaN,Frozen chopped onions labeled as: a) Kroger...,38006,81752 units,Food,Products contain onions which were recalled fo...,20160409,F-1655-2016,The Pictsweet Company,20160713,TN,Terminated,20170803.0,Voluntary: Firm initiated
2,#10 Pictsweet Drive,NaN,20160914,Bells,Class I,"BEST IF USED BY: Aug 3 2016, Aug 5 2016, Nov 9...",United States,nationwide,74123,"Two or more of the following: Email, Fax, Lett...",NaN,"Pictsweet(R) Seasoned Summer Vegetables, NET W...",38006,"871,952 Cases Total",Food,Possible Listeria Monocytogenes contamination ...,20160506,F-2232-2016,The Pictsweet Company,20160921,TN,Terminated,20170803.0,Voluntary: Firm initiated


## 1. Null Value Check

In [2]:
nulls = df.isnull().sum()
null_pct = (nulls / len(df) * 100).round(1)
null_summary = pd.DataFrame({'nulls': nulls, 'pct': null_pct})
null_summary[null_summary['nulls'] > 0].sort_values('nulls', ascending=False)

,nulls,pct
results.more_code_info,1000,100.0
results.address_2,942,94.2
results.product_quantity,57,5.7
Sum of results.termination_date,45,4.5
results.code_info,18,1.8
results.postal_code,12,1.2
results.state,11,1.1


**Findings:**
- `results.more_code_info` — 100% null → drop entirely, no data at all
- `results.address_2` — 94.2% null → drop, not needed for analysis
- `results.termination_date` — 4.5% null → **expected**, these are the "Ongoing" recalls
  that haven't been terminated yet, not a data error
- `results.product_quantity`, `results.code_info`, `results.postal_code` — small % null,
  not used in core analysis, left as-is

In [3]:
df = df.drop(columns=['results.more_code_info', 'results.address_2'])
print(f"Shape after dropping empty columns: {df.shape}")

Shape after dropping empty columns: (1000, 22)


## 2. Duplicate Check

In [4]:
print(f"Full duplicate rows: {df.duplicated().sum()}")
print(f"Duplicate recall_number: {df['results.recall_number'].duplicated().sum()}")
print(f"Duplicate event_id: {df['Sum of results.event_id'].duplicated().sum()}")
print(f"Unique event_ids: {df['Sum of results.event_id'].nunique()} out of {len(df)} rows")

Full duplicate rows: 0
Duplicate recall_number: 0
Duplicate event_id: 277
Unique event_ids: 723 out of 1000 rows


In [5]:
# Investigate: are duplicate event_ids true duplicates or multi-product recall events?
dup_event = df[df['Sum of results.event_id'].duplicated(keep=False)].sort_values('Sum of results.event_id')
dup_event[['Sum of results.event_id','results.recall_number','results.recalling_firm',
           'results.classification','results.product_description']].head(6)

,Sum of results.event_id,results.recall_number,results.recalling_firm,results.classification,results.product_description
571,62019,F-1532-2012,Kerry Foods,Class II,"KERRY, MELOBLEND 737, CODE:QO-05358, CONTAINS ..."
572,62019,F-1503-2012,Kerry Foods,Class II,"KERRY, CULTURED NF BUTTERMILK 986, CODE 031001..."
927,62676,F-1882-2012,GH Foods CA LLC,Class I,"Asian Stir Fry, packaged under the following l..."
929,62676,F-1899-2012,GH Foods CA LLC,Class I,"Garden Highway Chef Essentials Celery, Onions,..."
928,62676,F-1905-2012,GH Foods CA LLC,Class I,Sweet and Sour Stir Fry packaged under the fol...
881,62707,F-2209-2012,"Garden-Fresh Foods, Inc.",Class I,Garden-Fresh Family Style Macaroni Salad\t2060...


**Finding:** No true duplicate rows (`recall_number` — the actual unique recall
identifier — has zero duplicates). The 277 rows sharing an `event_id` are **not errors** —
a single recall *event* (e.g., a contamination issue at one facility) can cover multiple
distinct products recalled together, each with its own unique `recall_number`.

**Methodology decision:** This analysis treats each row as one **product-level recall**
(the standard unit FDA itself uses for `recall_number`), meaning:
- **1,000 rows = 1,000 product-level recalls**
- Representing **723 unique recall events**

This is stated explicitly here so dashboard totals are unambiguous.

## 3. Missing / International Data Check

In [6]:
print("Country breakdown:")
print(df['results.country'].value_counts())
print(f"\nRows missing state: {df['results.state'].isnull().sum()}")
df[df['results.state'].isnull()][['results.recalling_firm','results.country']].head(12)

Country breakdown:
results.country
United States    988
Canada             5
Israel             4
Armenia            1
Chile              1
Taiwan             1
Name: count, dtype: int64

Rows missing state: 11


,results.recalling_firm,results.country
156,Greenbelt Greenhouse Ltd,Canada
340,Naturo Aid Pharmaceutical Inc.,Canada
380,"Riverside Natural Foods, Ltd.",Canada
549,Sequel Naturals Ltd,Canada
711,Artashes LLC,Armenia
987,Procesadora Aguas Claras LTDA.,Chile
990,ELITE CONFECTIONERY LTD,Israel
991,ELITE CONFECTIONERY LTD,Israel
992,ELITE CONFECTIONERY LTD,Israel
993,ELITE CONFECTIONERY LTD,Israel


**Finding:** All rows missing `state` are non-U.S. companies (Canada, Israel, Chile,
Taiwan, Armenia) — not a data error, just a field that doesn't apply to non-U.S. firms.

**Decision:** Since this dashboard's core research questions are about U.S. state-level
patterns, and international recalls are a small, inconsistent slice (1.2% of records),
**international records are excluded** from this analysis to keep state-level findings
clean and unambiguous.

In [7]:
before = len(df)
df = df[df['results.country'] == 'United States'].copy()
after = len(df)
print(f"Removed {before - after} international records.")
print(f"Remaining U.S. records: {after}")

Removed 12 international records.
Remaining U.S. records: 988


## 4. Data Type / Consistency Check

In [8]:
print("Current data types for date-related columns:")
date_cols = ['Sum of results.recall_initiation_date', 'Sum of results.report_date',
             'Sum of results.center_classification_date', 'Sum of results.termination_date']
print(df[date_cols].dtypes)

Current data types for date-related columns:
Sum of results.recall_initiation_date          int64
Sum of results.report_date                     int64
Sum of results.center_classification_date      int64
Sum of results.termination_date              float64
dtype: object


**Finding:** All four date columns are stored as plain numbers (e.g., `20160409`), not
as actual dates. This happened because the API returns dates as YYYYMMDD-formatted text,
and Power BI's export converted them to numbers rather than recognizing them as dates.
In this state, they can't be used for proper time-series sorting, date math, or a
year/month breakdown — they'd just be treated as arbitrary integers.

Additionally, `Sum of results.event_id` carries a "Sum of" prefix left over from Power BI's
default aggregation setting. This is a unique identifier, not a quantity — it should never
be summed, and the column will be renamed for clarity.

**Fix:** convert each date column from a YYYYMMDD integer/float into a proper pandas
datetime column, and rename the event_id column.

In [9]:
def to_date(series):
    # Handle both int and float (with NaN) inputs safely
    return pd.to_datetime(series.dropna().astype(int).astype(str), format='%Y%m%d', errors='coerce')

df['recall_initiation_date'] = to_date(df['Sum of results.recall_initiation_date'])
df['report_date'] = to_date(df['Sum of results.report_date'])
df['center_classification_date'] = to_date(df['Sum of results.center_classification_date'])

# termination_date has real nulls (ongoing recalls) — reindex to preserve them as NaT
df['termination_date'] = pd.to_datetime(
    df['Sum of results.termination_date'].dropna().astype(int).astype(str),
    format='%Y%m%d', errors='coerce'
).reindex(df.index)

df = df.rename(columns={'Sum of results.event_id': 'event_id'})
df = df.drop(columns=['Sum of results.recall_initiation_date', 'Sum of results.report_date',
                       'Sum of results.center_classification_date', 'Sum of results.termination_date'])

print(df[['recall_initiation_date','report_date','center_classification_date','termination_date']].dtypes)
print(f"\nSample converted dates:")
df[['recall_initiation_date','termination_date']].head(3)

recall_initiation_date        datetime64[us]
report_date                   datetime64[us]
center_classification_date    datetime64[us]
termination_date              datetime64[us]
dtype: object

Sample converted dates:


,recall_initiation_date,termination_date
0,2016-04-09,2017-08-03
1,2016-04-09,2017-08-03
2,2016-05-06,2017-08-03


**Verification:** dates are now proper `datetime64` objects — usable for time-series
charts, correct chronological sorting, and date-based filtering in both this notebook and
once loaded into Power BI (which will now recognize these as real dates instead of numbers).

In [10]:
print("Product type (should be 100% Food, confirming correct API endpoint):")
print(df['results.product_type'].value_counts())

print("\nClassification values:")
print(df['results.classification'].value_counts())

print("\nDate range check:")
print(f"{df['recall_initiation_date'].min().date()} to {df['recall_initiation_date'].max().date()}")

Product type (should be 100% Food, confirming correct API endpoint):
results.product_type
Food    988
Name: count, dtype: int64

Classification values:
results.classification
Class II     500
Class I      436
Class III     52
Name: count, dtype: int64

Date range check:
2008-07-07 to 2026-05-12


**Finding:** `product_type` is 100% "Food" — confirms we pulled the correct API endpoint
(Food Enforcement, not mixed with Drug data). Classification values are clean (only
Class I/II/III, no typos or unexpected categories). Date range (2008–2026) is plausible,
no obvious data entry errors.

## 5. Final Cleaning Summary

In [11]:
print("=== CLEANING SUMMARY ===")
print(f"Raw records pulled from API:        1,000")
print(f"After dropping empty columns:       1,000 (columns: 24 -> 22)")
print(f"After excluding international:      {len(df)}")
print(f"Final dataset ready for Power BI:   {len(df)} U.S. food recall records")

=== CLEANING SUMMARY ===
Raw records pulled from API:        1,000
After dropping empty columns:       1,000 (columns: 24 -> 22)
After excluding international:      988
Final dataset ready for Power BI:   988 U.S. food recall records


In [12]:
df.to_csv("data/data_cleaned.csv", index=False)
print("Saved: data/data_cleaned.csv")
print(f"Final shape: {df.shape}")

Saved: data/data_cleaned.csv
Final shape: (988, 22)


## Next Step
`data/data_cleaned.csv` is now ready to be imported into Power BI (Get Data → Text/CSV) to
build the dashboard — KPI cards, classification breakdown, state-level analysis, and
recall-reason analysis.